In [15]:
import ee
import json
import ipyleaflet
import pandas as pd
from pathlib import Path

from landcover_explorer.settings import Settings
from landcover_explorer.knowledgebase.biomass_stats import (
    import_biomass_layer,
    compute_agb_loss_by_driver,
)

settings = Settings()

In [16]:
credentials = ee.ServiceAccountCredentials(
    settings.google_earth_service_account, str(settings.google_earth_key)
)
ee.Initialize(credentials)

land_cover_dataset = ee.ImageCollection(settings.collection_id).select(
    settings.collection_band_name
)

In [17]:
with open(settings.gadm_path) as f:
    gadm = json.load(f)

iso_feature = next(f for f in gadm["features"] if f["properties"]["GID_0"] == "MYS")
geometry = ee.Geometry(iso_feature["geometry"])

centroid = geometry.centroid().getInfo()
coords = centroid['coordinates']  # [longitude, latitude]
center = [coords[1], coords[0]]  # ipyleaflet uses [lat, lon]

In [ ]:
# agb_2020 = import_biomass_layer(2020, geometry)

# vis = {"min": 0, "max": 500, "palette": ["white", "green"]}
# map_id = agb_2020.getMapId(vis)
# tile_url = map_id["tile_fetcher"].url_format  

# m = ipyleaflet.Map(center=center, zoom=4)
# m.add_layer(ipyleaflet.TileLayer(url=tile_url, name='Aboveground Biomass'))
# m.add_control(ipyleaflet.LayersControl())
# m

In [ ]:
result = compute_agb_loss_by_driver(land_cover_dataset, geometry, 2020)
result

In [ ]:
result = compute_agb_loss_by_driver(land_cover_dataset, geometry, 2020)
result

In [4]:
records = []
for year in range(2001, 2025):
    try:
        loss = compute_agb_loss_by_driver(land_cover_dataset, geometry, year)
    except ee.EEException:
        continue  # no AGB image for this year
    records.append({"year": year, **loss})

In [5]:
agb_loss_df = pd.DataFrame(records)

agb_loss_cumulative_df = agb_loss_df.copy()
agb_loss_cumulative_df[["agriculture_agb_Mt", "settlement_agb_Mt"]] = (
    agb_loss_df[["agriculture_agb_Mt", "settlement_agb_Mt"]].cumsum()
)

In [7]:
agb_loss_df

,year,agriculture_agb_Mt,settlement_agb_Mt
0,2007,2.836326,4.632045
1,2010,5.280888,7.183944
2,2015,7.288809,12.101071
3,2016,7.977626,13.777033
4,2017,8.466390,15.223813
5,2018,9.244849,16.686108
6,2019,10.080607,17.916989
7,2020,11.571493,19.131120
8,2021,14.504538,20.462575
9,2022,18.426837,21.897824


In [8]:
agb_loss_cumulative_df

,year,agriculture_agb_Mt,settlement_agb_Mt
0,2007,2.836326,4.632045
1,2010,8.117214,11.815988
2,2015,15.406023,23.917060
3,2016,23.383648,37.694092
4,2017,31.850038,52.917905
5,2018,41.094887,69.604013
6,2019,51.175494,87.521002
7,2020,62.746987,106.652122
8,2021,77.251525,127.114697
9,2022,95.678363,149.012521


In [14]:
import plotly.express as px

agb_loss_long_df = agb_loss_df.melt(
    id_vars="year",
    value_vars=["agriculture_agb_Mt", "settlement_agb_Mt"],
    var_name="Driver",
    value_name="AGB Loss (Mt)",
)
agb_loss_long_df["Driver"] = agb_loss_long_df["Driver"].map({
    "agriculture_agb_Mt": "Agriculture",
    "settlement_agb_Mt": "Settlements",
})

fig = px.bar(
    agb_loss_long_df,
    x="year",
    y="AGB Loss (Mt)",
    color="Driver",
    barmode="stack",
    opacity=0.8,
    title="Aboveground Biomass Loss by Driver",
    template="plotly_dark",
    height=300,
)
fig.update_layout(
    margin=dict(l=40, r=160, t=60, b=60),
    legend=dict(
        orientation="v", x=1.02, y=1, xanchor="left", yanchor="top",
        font=dict(size=10), bgcolor="rgba(0,0,0,0)",
        title=dict(text="Driver", font=dict(size=11)),
    ),
)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hovertemplate': 'Driver=Agriculture<br>year=%{x}<br>AGB Loss (Mt)=%{y}<extra></extra>',
              'legendgroup': 'Agriculture',
              'marker': {'color': '#636efa', 'opacity': 0.8, 'pattern': {'shape': ''}},
              'name': 'Agriculture',
              'orientation': 'v',
              'showlegend': True,
              'textposition': 'auto',
              'type': 'bar',
              'x': {'bdata': '1wfaB98H4AfhB+IH4wfkB+UH5gc=', 'dtype': 'i2'},
              'xaxis': 'x',
              'y': {'bdata': ('9Q6xr8uwBkCtkKQUoR8VQII64HW9Jx' ... 'mvmiQnQCWrpthSAi1Alj3sOEVtMkA='),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'hovertemplate': 'Driver=Settlements<br>year=%{x}<br>AGB Loss (Mt)=%{y}<extra></extra>',
              'legendgroup': 'Settlements',
              'marker': {'color': '#EF553B', 'opacity': 0.8, 'pattern': {'shape': ''}},
              'name': 'Settlements',
              'orientation': 'v',
              'showlegend': True,
              'textposition': 'auto',
              'type': 'bar',
              'x': {'bdata': '1wfaB98H4AfhB+IH4wfkB+UH5gc=', 'dtype': 'i2'},
              'xaxis': 'x',
              'y': {'bdata': ('PmomvjaHEkDjSH+3W7wcQNuYrZu/My' ... '4TkSEzQE80u1RrdjRAh5UIzdflNUA='),
                    'dtype': 'f8'},
              'yaxis': 'y'}],
    'layout': {'barmode': 'stack',
               'height': 300,
               'legend': {'bgcolor': 'rgba(0,0,0,0)',
                          'font': {'size': 10},
                          'orientation': 'v',
                          'title': {'font': {'size': 11}, 'text': 'Driver'},
                          'tracegroupgap': 0,
                          'x': 1.02,
                          'xanchor': 'left',
                          'y': 1,
                          'yanchor': 'top'},
               'margin': {'b': 60, 'l': 40, 'r': 160, 't': 60},
               'template': '...',
               'title': {'text': 'Aboveground Biomass Loss by Driver'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'year'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'AGB Loss (Mt)'}}}
})

In [15]:
agb_loss_cumulative_df = agb_loss_cumulative_df.melt(
    id_vars="year",
    value_vars=["agriculture_agb_Mt", "settlement_agb_Mt"],
    var_name="Driver",
    value_name="AGB Loss (Mt)",
)
agb_loss_long_df["Driver"] = agb_loss_long_df["Driver"].map({
    "agriculture_agb_Mt": "Agriculture",
    "settlement_agb_Mt": "Settlements",
})

fig = px.bar(
    agb_loss_cumulative_df,
    x="year",
    y="AGB Loss (Mt)",
    color="Driver",
    barmode="stack",
    opacity=0.8,
    title="Cumulative Aboveground Biomass Loss by Driver",
    template="plotly_dark",
    height=300,
)
fig.update_layout(
    margin=dict(l=40, r=160, t=60, b=60),
    legend=dict(
        orientation="v", x=1.02, y=1, xanchor="left", yanchor="top",
        font=dict(size=10), bgcolor="rgba(0,0,0,0)",
        title=dict(text="Driver", font=dict(size=11)),
    ),
)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'hovertemplate': 'Driver=agriculture_agb_Mt<br>year=%{x}<br>AGB Loss (Mt)=%{y}<extra></extra>',
              'legendgroup': 'agriculture_agb_Mt',
              'marker': {'color': '#636efa', 'opacity': 0.8, 'pattern': {'shape': ''}},
              'name': 'agriculture_agb_Mt',
              'orientation': 'v',
              'showlegend': True,
              'textposition': 'auto',
              'type': 'bar',
              'x': {'bdata': '1wfaB98H4AfhB+IH4wfkB+UH5gc=', 'dtype': 'i2'},
              'xaxis': 'x',
              'y': {'bdata': ('9Q6xr8uwBkAUjD52AzwgQFWpLjHizy' ... 'VCnV9PQGmDf/wYUFNAzpK6SmrrV0A='),
                    'dtype': 'f8'},
              'yaxis': 'y'},
             {'hovertemplate': 'Driver=settlement_agb_Mt<br>year=%{x}<br>AGB Loss (Mt)=%{y}<extra></extra>',
              'legendgroup': 'settlement_agb_Mt',
              'marker': {'color': '#EF553B', 'opacity': 0.8, 'pattern': {'shape': ''}},
              'name': 'settlement_agb_Mt',
              'orientation': 'v',
              'showlegend': True,
              'textposition': 'auto',
              'type': 'bar',
              'x': {'bdata': '1wfaB98H4AfhB+IH4wfkB+UH5gc=', 'dtype': 'i2'},
              'xaxis': 'x',
              'y': {'bdata': ('PmomvjaHEkCQ2dI6yaEnQDY5QGvE6j' ... '9cvKlaQH+2TjFXx19A8G1IkmagYkA='),
                    'dtype': 'f8'},
              'yaxis': 'y'}],
    'layout': {'barmode': 'stack',
               'height': 300,
               'legend': {'bgcolor': 'rgba(0,0,0,0)',
                          'font': {'size': 10},
                          'orientation': 'v',
                          'title': {'font': {'size': 11}, 'text': 'Driver'},
                          'tracegroupgap': 0,
                          'x': 1.02,
                          'xanchor': 'left',
                          'y': 1,
                          'yanchor': 'top'},
               'margin': {'b': 60, 'l': 40, 'r': 160, 't': 60},
               'template': '...',
               'title': {'text': 'Cumulative Aboveground Biomass Loss by Driver'},
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'year'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'AGB Loss (Mt)'}}}
})

---

## Calculate total AGB / forest area per state

In [13]:
import ee
import ipyleaflet
import pandas as pd

from landcover_explorer.knowledgebase.biomass_stats import import_biomass_layer
from landcover_explorer.knowledgebase.gee_tiles_preprocess import (
    _compute_mask,
    extract_admin1_metric_centroids,
    FOREST_CODES,
)

from landcover_explorer.shiny.app_helpers import build_loss_markers

from landcover_explorer.settings import Settings

In [2]:
def compute_agb_and_forest_area_by_state(land_cover_dataset, select_year, select_country, tile_resolution):
    """Average AGB density (Mg/ha) within forest pixels, per admin-1 state.

    Computed as total AGB stock divided by total forest area within each state boundary.
    States with no forest area get a null density rather than a divide-by-zero.
    """
    admin1_regions = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(
        ee.Filter.eq("ADM0_NAME", select_country)
    )
    geometry = admin1_regions.geometry()

    agb_density = import_biomass_layer(select_year, geometry)  # Mg/ha
    forest_mask = _compute_mask(land_cover_dataset, geometry, select_year, FOREST_CODES)

    agb_mass = agb_density.multiply(ee.Image.pixelArea().divide(10_000)).updateMask(forest_mask)  # Mg per pixel
    forest_area = ee.Image.pixelArea().updateMask(forest_mask)  # m² per pixel

    combined = ee.Image.cat([
        agb_mass.rename("agb_Mg"),
        forest_area.rename("forest_area_m2"),
    ])
    admin1_totals = combined.reduceRegions(
        collection=admin1_regions,
        reducer=ee.Reducer.sum(),
        scale=tile_resolution,
        tileScale=4,
    )

    def _agb_per_forest_area(feature):
        forest_area_ha = ee.Number(feature.get("forest_area_m2")).divide(10_000)
        agb_Mg = ee.Number(feature.get("agb_Mg"))
        density = ee.Algorithms.If(forest_area_ha.gt(0), agb_Mg.divide(forest_area_ha), None)
        return feature.set("agb_density_Mg_per_ha", density)

    return admin1_totals.map(_agb_per_forest_area).select(["ADM1_NAME", "agb_density_Mg_per_ha"])

In [3]:
settings = Settings()

credentials = ee.ServiceAccountCredentials(
    settings.google_earth_service_account, str(settings.google_earth_key)
)
ee.Initialize(credentials)

In [4]:
MODIS_TILE_RESOLUTION = settings.collection_resolution
dataset = ee.ImageCollection(settings.collection_id)
igbp_land_cover = dataset.select(settings.collection_band_name)

admin1_density = compute_agb_and_forest_area_by_state(igbp_land_cover, 2020, "Malaysia", MODIS_TILE_RESOLUTION)


In [6]:
records = [
    {
        "state": f["properties"]["ADM1_NAME"],
        "agb_density_Mg_per_ha": f["properties"]["agb_density_Mg_per_ha"],
    }
    for f in admin1_density.getInfo()["features"]
]
df = pd.DataFrame(records)

In [9]:
agb_density_centroids = extract_admin1_metric_centroids(admin1_density, "agb_density_Mg_per_ha")

In [21]:
agb_density_markers = build_loss_markers(agb_density_centroids, "#2E8B57", "AGBDensity_Malaysia_2020", "agb_density_Mg_per_ha", 1, "Mg/ha")

In [23]:
m = ipyleaflet.Map(center=[3.82, 109.71], zoom=6)
m.add_layer(agb_density_markers)
m.add_control(ipyleaflet.LayersControl())
m

Map(center=[3.82, 109.71], controls=(ZoomControl(options=['position', 'zoom_in_text', 'zoom_in_title', 'zoom_o…